## **Evaluate Input Interventions**

This notebook evaluates the controlled input interventions created in `04.1_create_input_interventions.ipynb` using the trained TF-IDF logistic regression classifier.

The objective is to measure how model predictions change when specific information sources are systematically modified:

1. the supplied target identity,
2. explicit candidate mentions in the retrieved context posts, and
3. label-correlated lexical cues.

All intervention datasets were constructed from the held-out human-annotated test set. Any data-driven intervention rules were derived exclusively from the training data before being applied to the test set.

The TF-IDF model is loaded in its previously trained and frozen state.

Predictions on each modified input are compared with predictions on the corresponding original test example. For label-preserving interventions, changes in classification performance are also evaluated against the human gold labels. Target swapping is treated separately because the original stance label is no longer a valid gold label after changing the target.

---

### **1. Setup**

Load the libraries and define the paths to the fixed human test set, the frozen TF-IDF model, and the previously generated intervention datasets.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score


DATA_DIR = Path("../data/preprocessed")
INTERVENTION_DIR = Path("../data/interventions")
MODEL_DIR = Path("../models/tfidf_logreg")

HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"
MODEL_PATH = MODEL_DIR / "model.joblib"
RESULTS_DIR = Path("../results/tfidf_interventions")

In [2]:
# Collect paths for all intervention variants.
intervention_paths = {
    "target_masked": (INTERVENTION_DIR / "human_test_target_masked.parquet"),
    "target_swapped": (INTERVENTION_DIR / "human_test_target_swapped.parquet"),
    "candidate_mentions_masked": (INTERVENTION_DIR / "human_test_candidate_mentions_masked.parquet"),
    "lexical_cues_top10_masked": (INTERVENTION_DIR / "human_test_lexical_cues_top10_masked.parquet"),
    "lexical_cues_top25_masked": (INTERVENTION_DIR / "human_test_lexical_cues_top25_masked.parquet")
}

for seed in range(1, 6):
    intervention_paths[f"candidate_mentions_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_candidate_mentions_control_seed{seed}.parquet")

    intervention_paths[f"lexical_cues_top10_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_lexical_cues_top10_control_seed{seed}.parquet")

    intervention_paths[f"lexical_cues_top25_control_seed{seed}"] = (INTERVENTION_DIR / f"human_test_lexical_cues_top25_control_seed{seed}.parquet")

---

### **Load the fixed test set and frozen model**

The original human-annotated test set provides the reference examples and gold labels. The previously saved TF-IDF pipeline is loaded without modification and will be used for all original and intervention inputs.

In [3]:
human_test = pd.read_parquet(HUMAN_TEST_PATH)
tfidf_logreg = joblib.load(MODEL_PATH)

print(f"Human test examples: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")

Human test examples: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']


---

### **2. Inspect the Frozen TF-IDF Model**

Before applying the model to the test interventions, inspect the loaded pipeline and verify the label mapping used during training.

The intervention placeholders are also checked against the frozen TF-IDF vocabulary. This ensures that masking removes the selected information without unintentionally introducing placeholder tokens that already have learned model weights.

In [4]:
# Extract the fitted vectorizer and classifier from the frozen pipeline.
vectorizer = tfidf_logreg.named_steps["tfidf"]
classifier = tfidf_logreg.named_steps["classifier"]

# Inspect key properties of the fitted TF-IDF representation and classifier.
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Classifier classes: {classifier.classes_.tolist()}")
print(f"Lowercase: {vectorizer.lowercase}")
print(f"N-gram range: {vectorizer.ngram_range}")
print(f"Minimum document frequency: {vectorizer.min_df}")

Vocabulary size: 203,461
Classifier classes: [0, 1, 2]
Lowercase: True
N-gram range: (1, 2)
Minimum document frequency: 2


In [5]:
# Define the label mapping used throughout the evaluation.
LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "Neither": 2,
}

ID2LABEL = {
    value: key
    for key, value in LABEL2ID.items()
}

LABEL_ORDER = ["Against", "Favor", "Neither"]

In [6]:
# Verify that the classifier's internal class order matches the expected label mapping.
assert classifier.classes_.tolist() == list(ID2LABEL.keys())

print(
    "Classifier label order:",
    [ID2LABEL[class_id] for class_id in classifier.classes_],
)

Classifier label order: ['Against', 'Favor', 'Neither']


---

### **Verify the frozen context-masking token**

All context-level masking interventions were created with the synthetic token `requ`, which was selected and frozen before intervention generation in `04_select_masking_token.ipynb`.

For the TF-IDF model, the token must not overlap with any feature in the frozen vocabulary. Target masking is treated separately and uses an empty target string rather than a synthetic token.

In [7]:
# Ensure that the mask token is out-of-vocabulary for the fitted TF-IDF model.
CONTEXT_MASK_TOKEN = "requ"

mask_features = vectorizer.build_analyzer()(CONTEXT_MASK_TOKEN)

vocabulary_overlap = [
    feature
    for feature in mask_features
    if feature in vectorizer.vocabulary_
]

print("TF-IDF analyzer:", mask_features)
print("Vocabulary overlap:", vocabulary_overlap)

TF-IDF analyzer: ['requ']
Vocabulary overlap: []


---

### **3. Load Intervention Datasets**

Load all previously generated intervention datasets. Each dataset was verified to contain the same held-out test examples in the same row order as the original test set, allowing predictions to be compared pairwise with the corresponding original input.

In [8]:
intervention_data = {
    name: pd.read_parquet(path)
    for name, path in intervention_paths.items()
}

print(f"Loaded {len(intervention_data)} intervention datasets.")
for name in intervention_data:
    print(name)

Loaded 20 intervention datasets.
target_masked
target_swapped
candidate_mentions_masked
lexical_cues_top10_masked
lexical_cues_top25_masked
candidate_mentions_control_seed1
lexical_cues_top10_control_seed1
lexical_cues_top25_control_seed1
candidate_mentions_control_seed2
lexical_cues_top10_control_seed2
lexical_cues_top25_control_seed2
candidate_mentions_control_seed3
lexical_cues_top10_control_seed3
lexical_cues_top25_control_seed3
candidate_mentions_control_seed4
lexical_cues_top10_control_seed4
lexical_cues_top25_control_seed4
candidate_mentions_control_seed5
lexical_cues_top10_control_seed5
lexical_cues_top25_control_seed5


---

### **4. Construct Model Inputs**

Reconstruct the text input for the original test set and all intervention datasets using exactly the same input-construction functions as during TF-IDF training.

Keeping the input construction unchanged ensures that any prediction differences are caused by the interventions rather than by differences in preprocessing or formatting.

In [9]:
# Combine all valid context posts into a single text
def build_context_text(context_posts):
    posts = [
        post["Content"]
        for post in context_posts
        if isinstance(post["Content"], str)
        and post["Content"].strip()
    ]

    return " POST_SEP ".join(posts)


# Construct the final model input from target entity and context
def build_model_input(row):
    context_text = build_context_text(row["ContextPosts"])

    return (
        f"{row['TargetEntity']} TARGET_SEP "
        f"{context_text}"
    )

In [10]:
# Apply the same input-construction function to the original and all intervention variants.
original_texts = human_test.apply(
    build_model_input,
    axis=1,
)

intervention_texts = {
    name: df.apply(build_model_input, axis=1)
    for name, df in intervention_data.items()
}

---

### **5. Generate Original Test Predictions**

Generate predictions and class probabilities for the unchanged human-annotated test set.

These outputs serve as the reference for all subsequent intervention comparisons. The frozen model is applied to the human-annotated test set without any refitting or adaptation.

In [11]:
original_pred_ids = tfidf_logreg.predict(original_texts)
original_probabilities = tfidf_logreg.predict_proba(original_texts)

original_predictions = np.array([
    ID2LABEL[pred_id]
    for pred_id in original_pred_ids
])

print(f"Predictions: {len(original_predictions):,}")
print(f"Probability matrix: {original_probabilities.shape}")

Predictions: 890
Probability matrix: (890, 3)


---

### **6. Evaluate Original Test Performance**

Evaluate the unchanged human-annotated test set to establish the TF-IDF reference performance.

Macro-F1 is used as the main overall metric because the stance classes are imbalanced. Class-specific precision, recall, and F1-scores are additionally reported to inspect performance differences between stance classes.

In [12]:
y_true = human_test["StanceLabel"].to_numpy()

original_macro_f1 = f1_score(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    average="macro",
)

print(f"Original Macro-F1: {original_macro_f1:.4f}")

Original Macro-F1: 0.7348


In [13]:
# Compute the reference classification report for the unmodified test set.
original_report = classification_report(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)

original_report = pd.DataFrame(original_report).T

original_report

,precision,recall,f1-score,support
Against,0.828694,0.879545,0.853363,440.000000
Favor,0.757282,0.663830,0.707483,235.000000
Neither,0.640553,0.646512,0.643519,215.000000
accuracy,0.766292,0.766292,0.766292,0.766292
macro avg,0.742176,0.729962,0.734788,890.000000
weighted avg,0.764388,0.766292,0.764151,890.000000


The frozen TF-IDF model achieves a macro-F1 of 0.735 and an accuracy of 0.766 on the unmodified human-annotated test set. Performance is strongest for Against (F1 = 0.853), followed by Favor (0.707), while Neither is the most difficult class (0.644).

---

### **7. Generate Intervention Predictions**

Apply the same frozen TF-IDF model to every intervention dataset.

For each intervention, store both the predicted stance labels and the corresponding class probabilities. These outputs will later be compared pairwise with the predictions on the unchanged test inputs.

In [14]:
intervention_predictions = {}
intervention_probabilities = {}

# Apply the frozen TF-IDF model to all intervention variants and store both predicted labels and class probabilities for later comparison.
for name, texts in intervention_texts.items():

    pred_ids = tfidf_logreg.predict(texts)
    probabilities = tfidf_logreg.predict_proba(texts)

    predictions = np.array([
        ID2LABEL[pred_id]
        for pred_id in pred_ids
    ])

    intervention_predictions[name] = predictions
    intervention_probabilities[name] = probabilities

---

### **8. Compare Intervention Outcomes**

Compare each intervention with the predictions on the unchanged test inputs.

For label-preserving interventions, Macro-F1 is calculated against the human gold labels and its change relative to the original performance is reported. The prediction flip rate measures the proportion of examples for which the predicted stance label changes after the intervention.

Target swapping is evaluated only through prediction changes because changing the target invalidates the original gold stance label.

In [15]:
# Compare each intervention with the original predictions using macro-F1 change and prediction flip rate.
comparison_rows = [
    {
        "condition": "original",
        "macro_f1": original_macro_f1,
        "delta_macro_f1": 0.0,
        "flip_rate": 0.0,
    }
]

for name, predictions in intervention_predictions.items():

    flip_rate = np.mean(
        predictions != original_predictions
    )

    # The target-swap condition has no valid gold labels, so only prediction changes are evaluated for this intervention.
    if name == "target_swapped":
        macro_f1 = np.nan
        delta_macro_f1 = np.nan

    else:
        macro_f1 = f1_score(
            y_true,
            predictions,
            labels=LABEL_ORDER,
            average="macro",
        )

        delta_macro_f1 = (
            macro_f1 - original_macro_f1
        )

    comparison_rows.append(
        {
            "condition": name,
            "macro_f1": macro_f1,
            "delta_macro_f1": delta_macro_f1,
            "flip_rate": flip_rate,
        }
    )

comparison_results = pd.DataFrame(comparison_rows)

comparison_results

,condition,macro_f1,delta_macro_f1,flip_rate
0,original,0.734788,0.000000,0.000000
1,target_masked,0.718143,-0.016646,0.052809
2,target_swapped,NaN,NaN,0.138202
3,candidate_mentions_masked,0.589169,-0.145619,0.275281
4,lexical_cues_top10_masked,0.732399,-0.002390,0.002247
5,lexical_cues_top25_masked,0.730924,-0.003864,0.003371
6,candidate_mentions_control_seed1,0.730008,-0.004780,0.011236
7,lexical_cues_top10_control_seed1,0.732971,-0.001817,0.003371
8,lexical_cues_top25_control_seed1,0.729910,-0.004878,0.005618
9,candidate_mentions_control_seed2,0.729059,-0.005729,0.014607


The TF-IDF classifier shows limited sensitivity to masking the supplied target, but substantial sensitivity to explicit candidate mentions in the retrieved context. Candidate-mention masking reduces macro-F1 by approximately 0.146 and changes 27.5% of predictions, whereas matched random-removal controls have only minor effects. In contrast, masking the identified top-10 or top-25 lexical cues produces changes comparable to their controls, providing little evidence that these cue sets substantially drive the classifier's predictions.

---

### **9. Compare Masking Interventions with Matched Controls**

Aggregate the five matched-control runs for each masking intervention.

The control conditions replace approximately the same number of tokens as the corresponding masking intervention, but at matched non-target locations. Comparing the actual intervention with the average control effect helps distinguish reliance on the selected information from the general effect of modifying the same amount of input text.

In [16]:
control_groups = {
    "candidate_mentions_masked": "candidate_mentions_control_",
    "lexical_cues_top10_masked": "lexical_cues_top10_control_",
    "lexical_cues_top25_masked": "lexical_cues_top25_control_",
}

control_comparison_rows = []

# Compare each targeted masking intervention with its five matched random-control runs using mean performance, variability, and effect differences.
for intervention, control_prefix in control_groups.items():

    intervention_row = comparison_results.loc[
        comparison_results["condition"] == intervention
    ].iloc[0]

    controls = comparison_results[
        comparison_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    control_comparison_rows.append(
        {
            "intervention": intervention,
            "intervention_macro_f1": intervention_row["macro_f1"],
            "control_macro_f1_mean": controls["macro_f1"].mean(),
            "control_macro_f1_sd": controls["macro_f1"].std(),
            "macro_f1_vs_control": (
                intervention_row["macro_f1"]
                - controls["macro_f1"].mean()
            ),
            "intervention_flip_rate": intervention_row["flip_rate"],
            "control_flip_rate_mean": controls["flip_rate"].mean(),
            "control_flip_rate_sd": controls["flip_rate"].std(),
            "flip_rate_vs_control": (
                intervention_row["flip_rate"]
                - controls["flip_rate"].mean()
            ),
        }
    )

control_comparison = pd.DataFrame(
    control_comparison_rows
)

control_comparison

,intervention,intervention_macro_f1,control_macro_f1_mean,control_macro_f1_sd,macro_f1_vs_control,intervention_flip_rate,control_flip_rate_mean,control_flip_rate_sd,flip_rate_vs_control
0,candidate_mentions_masked,0.589169,0.732252,0.003251,-0.143083,0.275281,0.013258,0.003015,0.262022
1,lexical_cues_top10_masked,0.732399,0.732986,0.000562,-0.000587,0.002247,0.001798,0.001005,0.000449
2,lexical_cues_top25_masked,0.730924,0.731225,0.001139,-0.000301,0.003371,0.003820,0.001281,-0.000449


Masking explicit candidate mentions caused a substantial decrease in macro-F1 relative to matched random-removal controls (0.589 vs. 0.732) and increased the prediction flip rate from approximately 1.3% under the controls to 27.5%. In contrast, masking the top-10 and top-25 label-associated lexical cues produced performance and prediction changes that were nearly identical to their matched controls.

---

### **10. Analyze Changes in Predicted Probabilities**

Prediction flips capture only cases in which an intervention changes the final predicted stance label. Smaller changes in model confidence can occur even when the predicted label remains unchanged.

For each intervention, we therefore measure:

- the mean change in probability assigned to the originally predicted class, and
- the mean absolute change across all three class probabilities.

Negative changes in the original-class probability indicate that the intervention weakens the model's original decision.

In [17]:
original_pred_indices = np.array([
    LABEL2ID[label]
    for label in original_predictions
])

row_indices = np.arange(len(human_test))

original_predicted_class_prob = original_probabilities[
    row_indices,
    original_pred_indices,
]

probability_rows = []

# Measure how each intervention changes the model's predicted probabilities relative to the original, unmodified inputs.
for name, probabilities in intervention_probabilities.items():

    intervention_original_class_prob = probabilities[
        row_indices,
        original_pred_indices,
    ]

    original_class_probability_change = (
        intervention_original_class_prob
        - original_predicted_class_prob
    )

    mean_absolute_probability_change = np.abs(
        probabilities - original_probabilities
    ).mean()

    probability_rows.append(
        {
            "condition": name,
            "mean_original_class_probability_change":
                original_class_probability_change.mean(),
            "mean_absolute_probability_change":
                mean_absolute_probability_change,
        }
    )

probability_results = pd.DataFrame(probability_rows)

probability_results

,condition,mean_original_class_probability_change,mean_absolute_probability_change
0,target_masked,-0.040443,0.039179
1,target_swapped,-0.092122,0.085635
2,candidate_mentions_masked,-0.190273,0.161172
3,lexical_cues_top10_masked,-0.000237,0.002435
4,lexical_cues_top25_masked,0.000378,0.004548
5,candidate_mentions_control_seed1,0.003515,0.009827
6,lexical_cues_top10_control_seed1,-0.000997,0.001540
7,lexical_cues_top25_control_seed1,-0.000853,0.002869
8,candidate_mentions_control_seed2,0.003047,0.010237
9,lexical_cues_top10_control_seed2,-0.000928,0.001582


In [18]:
probability_control_rows = []

for intervention, control_prefix in control_groups.items():

    intervention_row = probability_results.loc[
        probability_results["condition"] == intervention
    ].iloc[0]

    controls = probability_results[
        probability_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    probability_control_rows.append(
        {
            "intervention": intervention,

            "intervention_original_class_prob_change":
                intervention_row[
                    "mean_original_class_probability_change"
                ],

            "control_original_class_prob_change_mean":
                controls[
                    "mean_original_class_probability_change"
                ].mean(),

            "control_original_class_prob_change_sd":
                controls[
                    "mean_original_class_probability_change"
                ].std(),

            "intervention_mean_abs_prob_change":
                intervention_row[
                    "mean_absolute_probability_change"
                ],

            "control_mean_abs_prob_change_mean":
                controls[
                    "mean_absolute_probability_change"
                ].mean(),

            "control_mean_abs_prob_change_sd":
                controls[
                    "mean_absolute_probability_change"
                ].std(),
        }
    )

probability_control_comparison = pd.DataFrame(
    probability_control_rows
)

probability_control_comparison

,intervention,intervention_original_class_prob_change,control_original_class_prob_change_mean,control_original_class_prob_change_sd,intervention_mean_abs_prob_change,control_mean_abs_prob_change_mean,control_mean_abs_prob_change_sd
0,candidate_mentions_masked,-0.190273,0.003382,0.000362,0.161172,0.009978,0.000414
1,lexical_cues_top10_masked,-0.000237,-0.001074,0.000160,0.002435,0.001602,0.000067
2,lexical_cues_top25_masked,0.000378,-0.001183,0.000247,0.004548,0.002831,0.000124


Probability-based results reinforce the performance-based findings. Masking explicit candidate mentions substantially reduced the probability assigned to the model's original prediction and strongly altered the overall class-probability distribution, whereas matched random removals produced only minor changes. In contrast, masking the top-10 and top-25 lexical cue sets resulted in only small probability shifts, broadly consistent with the corresponding controls.

---

### **11. Analyze Intervention Effects by Target**

Evaluate whether intervention effects differ between the two political targets.

All examples are grouped by their original target in the human-annotated test set. For label-preserving interventions, target-specific Macro-F1 and prediction flip rates are reported. For target swapping, only behavioral changes are evaluated because the original gold label is no longer valid after changing the target.

The preceding dataset analysis showed a strong association between target and stance label in both the training and human-annotated test data. Target-specific Macro-F1 is therefore retained as a performance measure, but absolute Macro-F1 values are not interpreted as directly comparable measures of difficulty across targets. The main focus is on within-target changes relative to the original predictions.

In [19]:
target_rows = []

# Recompute intervention effects separately for each target to assess whether model sensitivity differs between Trump and Harris.
for target in human_test["TargetEntity"].unique():

    target_mask = (
        human_test["TargetEntity"].to_numpy() == target
    )

    target_indices = np.where(target_mask)[0]

    target_y_true = y_true[target_mask]
    target_original_predictions = original_predictions[target_mask]

    original_target_macro_f1 = f1_score(
        target_y_true,
        target_original_predictions,
        labels=LABEL_ORDER,
        average="macro",
    )

    original_target_class_prob = original_probabilities[
        target_indices,
        original_pred_indices[target_indices],
    ]

    target_rows.append(
        {
            "target": target,
            "condition": "original",
            "macro_f1": original_target_macro_f1,
            "delta_macro_f1": 0.0,
            "flip_rate": 0.0,
            "mean_original_class_probability_change": 0.0,
            "mean_absolute_probability_change": 0.0,
        }
    )

    for name, predictions in intervention_predictions.items():

        target_predictions = predictions[target_mask]
        target_probabilities = intervention_probabilities[name][
            target_indices
        ]

        flip_rate = np.mean(
            target_predictions
            != target_original_predictions
        )

        intervention_target_class_prob = target_probabilities[
            np.arange(len(target_indices)),
            original_pred_indices[target_indices],
        ]

        mean_original_class_probability_change = (
            intervention_target_class_prob
            - original_target_class_prob
        ).mean()

        mean_absolute_probability_change = np.abs(
            target_probabilities
            - original_probabilities[target_indices]
        ).mean()

        if name == "target_swapped":
            macro_f1 = np.nan
            delta_macro_f1 = np.nan

        else:
            macro_f1 = f1_score(
                target_y_true,
                target_predictions,
                labels=LABEL_ORDER,
                average="macro",
            )

            delta_macro_f1 = (
                macro_f1
                - original_target_macro_f1
            )

        target_rows.append(
            {
                "target": target,
                "condition": name,
                "macro_f1": macro_f1,
                "delta_macro_f1": delta_macro_f1,
                "flip_rate": flip_rate,
                "mean_original_class_probability_change":
                    mean_original_class_probability_change,
                "mean_absolute_probability_change":
                    mean_absolute_probability_change,
            }
        )

target_results = pd.DataFrame(target_rows)

In [20]:
# Display the original condition and the main targeted interventions only.
main_conditions = [
    "original",
    "target_masked",
    "target_swapped",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

target_results[
    target_results["condition"].isin(main_conditions)
].reset_index(drop=True)

,target,condition,macro_f1,delta_macro_f1,flip_rate,mean_original_class_probability_change,mean_absolute_probability_change
0,Trump,original,0.427360,0.000000,0.000000,0.000000,0.000000
1,Trump,target_masked,0.439789,0.012428,0.031461,-0.051181,0.039983
2,Trump,target_swapped,NaN,NaN,0.080899,-0.106618,0.080114
3,Trump,candidate_mentions_masked,0.420571,-0.006790,0.177528,-0.247559,0.172722
4,Trump,lexical_cues_top10_masked,0.427360,0.000000,0.000000,-0.000435,0.002302
5,Trump,lexical_cues_top25_masked,0.427360,0.000000,0.000000,0.000199,0.004307
6,Harris,original,0.533127,0.000000,0.000000,0.000000,0.000000
7,Harris,target_masked,0.551887,0.018760,0.074157,-0.029705,0.038375
8,Harris,target_swapped,NaN,NaN,0.195506,-0.077625,0.091155
9,Harris,candidate_mentions_masked,0.463039,-0.070088,0.373034,-0.132987,0.149622


---

### **12. Analyze Intervention Effects by Stance Class**

Examine whether intervention effects differ across the three stance classes.

For each label-preserving condition, precision, recall, and F1-score are calculated separately for `Against`, `Favor`, and `Neither`. Changes in class-specific F1 relative to the unchanged test inputs are reported to determine whether global performance changes are concentrated in particular stance classes.

Target swapping is excluded from this analysis because changing the target invalidates the original gold stance label.

In [21]:
class_rows = []

# Compute class-specific precision, recall, F1, and support for the original and all label-preserving intervention conditions.
label_preserving_predictions = {
    "original": original_predictions,
    **{
        name: predictions
        for name, predictions in intervention_predictions.items()
        if name != "target_swapped"
    },
}

for condition, predictions in label_preserving_predictions.items():

    report = classification_report(
        y_true,
        predictions,
        labels=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )

    for label in LABEL_ORDER:

        class_rows.append(
            {
                "condition": condition,
                "class": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1": report[label]["f1-score"],
                "support": report[label]["support"],
            }
        )

class_results = pd.DataFrame(class_rows)

In [22]:
# Compute class-specific F1 changes relative to the original predictions.
original_class_f1 = (
    class_results[
        class_results["condition"] == "original"
    ]
    .set_index("class")["f1"]
)

class_results["delta_f1"] = class_results.apply(
    lambda row: (
        row["f1"]
        - original_class_f1[row["class"]]
    ),
    axis=1,
)

In [23]:
# Display only the main label-preserving interventions.
main_label_preserving_conditions = [
    "original",
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

class_results[
    class_results["condition"].isin(
        main_label_preserving_conditions
    )
].reset_index(drop=True)

,condition,class,precision,recall,f1,support,delta_f1
0,original,Against,0.828694,0.879545,0.853363,440.0,0.000000
1,original,Favor,0.757282,0.663830,0.707483,235.0,0.000000
2,original,Neither,0.640553,0.646512,0.643519,215.0,0.000000
3,target_masked,Against,0.797521,0.877273,0.835498,440.0,-0.017865
4,target_masked,Favor,0.782609,0.612766,0.687351,235.0,-0.020132
5,target_masked,Neither,0.621622,0.641860,0.631579,215.0,-0.011940
6,candidate_mentions_masked,Against,0.848635,0.777273,0.811388,440.0,-0.041975
7,candidate_mentions_masked,Favor,0.934426,0.242553,0.385135,235.0,-0.322348
8,candidate_mentions_masked,Neither,0.429577,0.851163,0.570983,215.0,-0.072536
9,lexical_cues_top10_masked,Against,0.828326,0.877273,0.852097,440.0,-0.001266


The class-specific results show that the strongest intervention effect occurs for the **Favor** class after masking candidate mentions. Its F1-score drops from 0.707 to 0.385, mainly because recall decreases substantially. The effects on **Against** and **Neither** are smaller, while target masking causes only minor changes across all three classes. Masking the top-10 or top-25 lexical cues has almost no class-specific effect.

---

### **13. Audit Intervention Coverage**

Quantify how frequently each masking intervention actually modifies the human-annotated test inputs.

A small behavioral effect can only be interpreted as weak model reliance if the corresponding intervention affects a meaningful number of test examples. We therefore report the number and proportion of examples whose model input changes under each intervention.

In [24]:
# Measure intervention coverage by checking how many test examples actually differ from their original model input.
coverage_conditions = [
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

coverage_rows = []

for name in coverage_conditions:

    changed = (
        intervention_texts[name].to_numpy()
        != original_texts.to_numpy()
    )

    coverage_rows.append(
        {
            "condition": name,
            "changed_examples": changed.sum(),
            "total_examples": len(changed),
            "coverage_rate": changed.mean(),
        }
    )

coverage_results = pd.DataFrame(coverage_rows)

coverage_results

,condition,changed_examples,total_examples,coverage_rate
0,target_masked,890,890,1.000000
1,candidate_mentions_masked,843,890,0.947191
2,lexical_cues_top10_masked,126,890,0.141573
3,lexical_cues_top25_masked,275,890,0.308989


Intervention coverage differs substantially across conditions. Target masking modifies all test examples, while candidate-mention masking affects 94.7%. In contrast, the top-10 and top-25 lexical-cue interventions modify only 14.2% and 30.9% of examples, respectively. The small aggregate effects of lexical-cue masking should therefore be interpreted alongside their more limited coverage.

---

### **14. Analyze Effects Among Modified Examples**

The preceding analyses report behavioral changes across the complete test set. Because intervention coverage differs substantially between conditions, these aggregate effects can be diluted by examples that were not modified.

We therefore additionally report prediction flips and probability changes conditional on examples whose model input was actually changed. These conditional measures complement, rather than replace, the full-test results.

In [25]:
affected_rows = []

# Recompute intervention effects only for examples whose model input was actually changed by the corresponding intervention.
for name in coverage_conditions:

    changed = (
        intervention_texts[name].to_numpy()
        != original_texts.to_numpy()
    )

    predictions = intervention_predictions[name]
    probabilities = intervention_probabilities[name]

    affected_indices = np.where(changed)[0]

    affected_flip_rate = np.mean(
        predictions[changed]
        != original_predictions[changed]
    )

    affected_original_class_prob_change = (
        probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
        - original_probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
    ).mean()

    affected_mean_abs_prob_change = np.abs(
        probabilities[affected_indices]
        - original_probabilities[affected_indices]
    ).mean()

    affected_rows.append(
        {
            "condition": name,
            "affected_examples": changed.sum(),
            "affected_flip_rate": affected_flip_rate,
            "affected_original_class_prob_change":
                affected_original_class_prob_change,
            "affected_mean_abs_prob_change":
                affected_mean_abs_prob_change,
        }
    )

affected_results = pd.DataFrame(affected_rows)

affected_results

,condition,affected_examples,affected_flip_rate,affected_original_class_prob_change,affected_mean_abs_prob_change
0,target_masked,890,0.052809,-0.040443,0.039179
1,candidate_mentions_masked,843,0.290629,-0.200881,0.170158
2,lexical_cues_top10_masked,126,0.015873,-0.001672,0.017200
3,lexical_cues_top25_masked,275,0.010909,0.001223,0.014720


Even when restricting the analysis to examples that were actually modified, candidate-mention masking strongly affects predictions, whereas lexical-cue masking produces only minor changes. This suggests that the weak lexical-cue effects cannot be explained by limited intervention coverage alone.

---

### **15. Compare Conditional Effects with Matched Controls**

Compare each masking intervention with its matched-control variants among examples whose model input was actually modified.

Because intervention coverage differs substantially across intervention types, behavioral effects are calculated conditionally on the affected examples. The matched controls provide a like-for-like comparison with random modifications of comparable extent, and the five control runs are summarized by their mean and standard deviation.

In [26]:
# Restrict each targeted intervention and its matched controls to the examples actually modified by that specific condition.
conditional_rows = []

conditional_conditions = [
    name
    for name in intervention_predictions
    if (
        name == "candidate_mentions_masked"
        or name == "lexical_cues_top10_masked"
        or name == "lexical_cues_top25_masked"
        or "_control_seed" in name
    )
]

for name in conditional_conditions:

    changed = (
        intervention_texts[name].to_numpy()
        != original_texts.to_numpy()
    )

    affected_indices = np.where(changed)[0]

    predictions = intervention_predictions[name]
    probabilities = intervention_probabilities[name]

    flip_rate = np.mean(
        predictions[affected_indices]
        != original_predictions[affected_indices]
    )

    original_class_prob_change = (
        probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
        - original_probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
    ).mean()

    mean_abs_prob_change = np.abs(
        probabilities[affected_indices]
        - original_probabilities[affected_indices]
    ).mean()

    conditional_rows.append(
        {
            "condition": name,
            "affected_examples": len(affected_indices),
            "flip_rate": flip_rate,
            "mean_original_class_probability_change":
                original_class_prob_change,
            "mean_absolute_probability_change":
                mean_abs_prob_change,
        }
    )

conditional_results = pd.DataFrame(
    conditional_rows
)

conditional_results

,condition,affected_examples,flip_rate,mean_original_class_probability_change,mean_absolute_probability_change
0,candidate_mentions_masked,843,0.290629,-0.200881,0.170158
1,lexical_cues_top10_masked,126,0.015873,-0.001672,0.017200
2,lexical_cues_top25_masked,275,0.010909,0.001223,0.014720
3,candidate_mentions_control_seed1,843,0.011862,0.003710,0.010375
4,lexical_cues_top10_control_seed1,126,0.023810,-0.007046,0.010880
5,lexical_cues_top25_control_seed1,275,0.018182,-0.002762,0.009285
6,candidate_mentions_control_seed2,843,0.015421,0.003217,0.010808
7,lexical_cues_top10_control_seed2,126,0.007937,-0.006558,0.011177
8,lexical_cues_top25_control_seed2,275,0.007273,-0.004089,0.008813
9,candidate_mentions_control_seed3,843,0.009490,0.003903,0.009836


In [27]:
conditional_control_rows = []

for intervention, control_prefix in control_groups.items():

    intervention_row = conditional_results.loc[
        conditional_results["condition"] == intervention
    ].iloc[0]

    controls = conditional_results[
        conditional_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    conditional_control_rows.append(
        {
            "intervention": intervention,

            "intervention_flip_rate":
                intervention_row["flip_rate"],

            "control_flip_rate_mean":
                controls["flip_rate"].mean(),

            "control_flip_rate_sd":
                controls["flip_rate"].std(),

            "intervention_original_class_prob_change":
                intervention_row[
                    "mean_original_class_probability_change"
                ],

            "control_original_class_prob_change_mean":
                controls[
                    "mean_original_class_probability_change"
                ].mean(),

            "control_original_class_prob_change_sd":
                controls[
                    "mean_original_class_probability_change"
                ].std(),

            "intervention_mean_abs_prob_change":
                intervention_row[
                    "mean_absolute_probability_change"
                ],

            "control_mean_abs_prob_change_mean":
                controls[
                    "mean_absolute_probability_change"
                ].mean(),

            "control_mean_abs_prob_change_sd":
                controls[
                    "mean_absolute_probability_change"
                ].std(),
        }
    )

conditional_control_comparison = pd.DataFrame(
    conditional_control_rows
)

conditional_control_comparison

,intervention,intervention_flip_rate,control_flip_rate_mean,control_flip_rate_sd,intervention_original_class_prob_change,control_original_class_prob_change_mean,control_original_class_prob_change_sd,intervention_mean_abs_prob_change,control_mean_abs_prob_change_mean,control_mean_abs_prob_change_sd
0,candidate_mentions_masked,0.290629,0.013998,0.003183,-0.200881,0.003570,0.000382,0.170158,0.010534,0.000437
1,lexical_cues_top10_masked,0.015873,0.012698,0.007099,-0.001672,-0.007587,0.001131,0.017200,0.011317,0.000476
2,lexical_cues_top25_masked,0.010909,0.012364,0.004146,0.001223,-0.003830,0.000800,0.014720,0.009162,0.000401


Conditional on intervention exposure, candidate-mention masking produces substantially larger prediction and probability changes than its matched random controls. In contrast, the top-10 and top-25 lexical-cue interventions show flip rates close to the corresponding controls, with only small additional changes in the probability distributions. This indicates that the weak lexical-cue effects cannot be attributed to limited intervention coverage alone.

---

### **16. Analyze Correctness Transitions**

Prediction flips do not indicate whether an intervention harms or improves a previously made decision.

For each label-preserving main intervention, we therefore distinguish between predictions that remain correct, change from correct to incorrect, change from incorrect to correct, or remain incorrect relative to the human gold labels.

In [28]:
transition_conditions = [
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

original_correct = (
    original_predictions == y_true
)

transition_rows = []

for name in transition_conditions:

    intervention_correct = (
        intervention_predictions[name] == y_true
    )

    stable_correct = (
        original_correct & intervention_correct
    )

    correct_to_wrong = (
        original_correct & ~intervention_correct
    )

    wrong_to_correct = (
        ~original_correct & intervention_correct
    )

    stable_wrong = (
        ~original_correct & ~intervention_correct
    )

    transition_rows.append(
        {
            "condition": name,

            "stable_correct_n":
                stable_correct.sum(),

            "correct_to_wrong_n":
                correct_to_wrong.sum(),

            "wrong_to_correct_n":
                wrong_to_correct.sum(),

            "stable_wrong_n":
                stable_wrong.sum(),

            "correct_to_wrong_rate":
                correct_to_wrong.mean(),

            "wrong_to_correct_rate":
                wrong_to_correct.mean(),

            "net_correct_change":
                wrong_to_correct.sum()
                - correct_to_wrong.sum(),
        }
    )

transition_results = pd.DataFrame(
    transition_rows
)

transition_results

,condition,stable_correct_n,correct_to_wrong_n,wrong_to_correct_n,stable_wrong_n,correct_to_wrong_rate,wrong_to_correct_rate,net_correct_change
0,target_masked,655,27,13,195,0.030337,0.014607,-14
1,candidate_mentions_masked,523,159,59,149,0.178652,0.066292,-100
2,lexical_cues_top10_masked,680,2,0,208,0.002247,0.000000,-2
3,lexical_cues_top25_masked,679,3,0,208,0.003371,0.000000,-3


Candidate-mention masking produces the largest deterioration in prediction correctness, changing 159 originally correct predictions to incorrect ones while correcting 59 originally incorrect predictions, for a net loss of 100 correct predictions. Target masking has a substantially smaller net effect (-14), while the lexical-cue interventions change correctness only marginally (-2 and -3).

---

### **17. Inspect Prediction Transitions under Candidate Masking**

Candidate-mention masking produced by far the strongest behavioral and performance effect. As a descriptive follow-up analysis, we therefore examine this intervention in more detail by comparing the original predicted labels with the predictions obtained after masking explicit candidate references.

The transition matrix shows which predicted stance classes are preserved and which classes the model switches to after candidate information is removed.

In [29]:
# Count how predictions transition between stance classes after masking candidate mentions.
candidate_transition_counts = pd.crosstab(
    pd.Series(
        original_predictions,
        name="Original prediction",
    ),
    pd.Series(
        intervention_predictions[
            "candidate_mentions_masked"
        ],
        name="Candidate-masked prediction",
    ),
)

candidate_transition_counts = candidate_transition_counts.reindex(
    index=LABEL_ORDER,
    columns=LABEL_ORDER,
    fill_value=0,
)

candidate_transition_counts

Candidate-masked prediction,Against,Favor,Neither
Original prediction,,,
Against,368,1,98
Favor,35,60,111
Neither,0,0,217


In [30]:
candidate_transition_rates = (
    candidate_transition_counts
    .div(
        candidate_transition_counts.sum(axis=1),
        axis=0,
    )
)

candidate_transition_rates.round(3)

Candidate-masked prediction,Against,Favor,Neither
Original prediction,,,
Against,0.788,0.002,0.210
Favor,0.170,0.291,0.539
Neither,0.000,0.000,1.000


Candidate-mention masking causes a clear redistribution of predictions toward **Neither**. Only 29.1% of originally predicted Favor cases remain Favor, while 53.9% shift to Neither and 17.0% to Against. Against predictions are more stable, although 21.0% also shift to Neither. In contrast, all original Neither predictions remain unchanged. This transition pattern is consistent with the strong drop in Favor recall and the simultaneous increase in Neither recall after masking candidate mentions.

---

### **18. Save Evaluation Results**

Save the main TF-IDF intervention results for subsequent reporting and visualization.

In [31]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

comparison_results.to_csv(
    RESULTS_DIR / "overall_intervention_results.csv",
    index=False,
)

control_comparison.to_csv(
    RESULTS_DIR / "matched_control_results.csv",
    index=False,
)

target_results.to_csv(
    RESULTS_DIR / "target_specific_results.csv",
    index=False,
)

class_results.to_csv(
    RESULTS_DIR / "class_specific_results.csv",
    index=False,
)

coverage_results.to_csv(
    RESULTS_DIR / "intervention_coverage.csv",
    index=False,
)

conditional_control_comparison.to_csv(
    RESULTS_DIR / "conditional_control_results.csv",
    index=False,
)

transition_results.to_csv(
    RESULTS_DIR / "correctness_transitions.csv",
    index=False,
)

candidate_transition_counts.to_csv(
    RESULTS_DIR / "candidate_prediction_transitions.csv"
)

probability_results.to_csv(
    RESULTS_DIR / "probability_change_results.csv",
    index=False,
)

probability_control_comparison.to_csv(
    RESULTS_DIR / "probability_control_results.csv",
    index=False,
)

affected_results.to_csv(
    RESULTS_DIR / "affected_example_results.csv",
    index=False,
)

print(f"Saved TF-IDF intervention results to: {RESULTS_DIR}")

Saved TF-IDF intervention results to: ../results/tfidf_interventions


---

### **19. Summary of Main Findings**

The intervention analysis reveals clear differences in how strongly the TF-IDF logistic regression model relies on the tested information sources.

- **The supplied target has a measurable but comparatively limited influence on the model.** Masking the target decreases Macro-F1 from 0.7348 to 0.7181 and changes 5.3% of predictions. Swapping the target while keeping the retrieved posts unchanged produces a larger prediction flip rate of 13.8%, indicating some target sensitivity. Because the swapped inputs do not have valid gold labels, this intervention is interpreted only as a behavioral sensitivity test.

- **Explicit candidate references have by far the strongest effect.** Masking candidate mentions reduces Macro-F1 to 0.5892 and changes 27.5% of all predictions. Among the 843 examples whose input was actually modified, the flip rate reaches 29.1% and the probability assigned to the originally predicted class decreases by 0.201 on average. Matched random-control removals produce substantially smaller changes, indicating that the observed effect is substantially larger than the effect of comparable random text removal.

- **The candidate-reference effect is particularly strong for Favor predictions.** The Favor class F1 decreases from 0.7075 to 0.3851 after candidate masking. In the descriptive transition analysis, only 29.1% of original Favor predictions remain Favor, while 53.9% change to Neither. This suggests that explicit candidate-related information is especially important for the model's Favor predictions.

- **The selected label-correlated lexical cues show little evidence of strong model reliance.** The Top-10 and Top-25 cue interventions affect only 14.2% and 30.9% of test examples, respectively, and produce only very small changes in overall Macro-F1 and prediction behavior. Even when the analysis is restricted to affected examples, their effects remain close to those produced by matched random controls.

Overall, the TF-IDF model does not appear to rely equally on all potential shortcut sources. Its predictions are only moderately sensitive to the separately supplied target and show little dependence on the selected label-correlated lexical cues, whereas explicit candidate references in the retrieved posts provide a substantially stronger source of predictive information.